# Sales dashboard

An example notebook for `examples/`: charts of a made-up chain of stores in Java (`data/orders.csv`, written by `data/make_orders.py`). Each chart is shown here and kept as a PNG artifact of the run; the summary comes back as the run's output.

Parameters: `provinces` (empty: all six) and `weeks` (the last N of the 26).

In [ ]:
# Parameters: the caller's values replace these.
provinces = []
weeks = 26

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

import nbsandbox

OUT = os.environ["NBSANDBOX_OUTPUT_DIR"]  # files written here become the run's artifacts
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})


def save(fig, name):
    """Show a figure in the notebook and keep it as a PNG artifact."""
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight")
    plt.show()


orders = pd.read_csv("data/orders.csv", parse_dates=["week"])
stores = pd.read_csv("data/stores.csv")
sales = orders.merge(stores, on="store")

if provinces:
    unknown = set(provinces) - set(stores["province"])
    if unknown:
        raise ValueError(f"unknown provinces: {sorted(unknown)}")
    sales = sales[sales["province"].isin(provinces)]
first = sales["week"].max() - pd.Timedelta(weeks=weeks - 1)
sales = sales[sales["week"] >= first]
print(f"{len(sales):,} rows: {sales['store'].nunique()} stores in {sales['province'].nunique()} provinces, "
      f"{sales['week'].min():%d %b} to {sales['week'].max():%d %b %Y}")

## Weekly revenue by province

Thin lines are weeks; thick lines the 4-week average. The fasting month (weeks 8–11) dips, the holidays after it peak.

In [ ]:
weekly = sales.pivot_table(index="week", columns="province", values="revenue", aggfunc="sum") / 1000  # IDR millions
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, prov in enumerate(weekly.columns):
    color = f"C{i}"
    ax.plot(weekly.index, weekly[prov], color=color, alpha=0.3, linewidth=1)
    ax.plot(weekly.index, weekly[prov].rolling(4, min_periods=1).mean(), color=color, linewidth=2.2, label=prov)
ax.set_ylabel("revenue (IDR millions)")
ax.set_title("Weekly revenue by province")
ax.legend(ncol=3, frameon=False, loc="upper left")
save(fig, "weekly_revenue.png")

## Revenue by month and category

In [ ]:
monthly = (sales.assign(month=sales["week"].dt.to_period("M").astype(str))
           .pivot_table(index="month", columns="category", values="revenue", aggfunc="sum") / 1000)
fig, ax = plt.subplots(figsize=(9, 4))
bottom = None
for i, cat in enumerate(monthly.columns):
    ax.bar(monthly.index, monthly[cat], bottom=bottom, label=cat, color=f"C{i}", width=0.65)
    bottom = monthly[cat] if bottom is None else bottom + monthly[cat]
for x, total in zip(monthly.index, monthly.sum(axis=1)):
    ax.annotate(f"{total:,.0f}", (x, total), ha="center", va="bottom", fontsize=8, xytext=(0, 2), textcoords="offset points")
ax.set_ylabel("revenue (IDR millions)")
ax.set_title("Revenue by month and category")
ax.legend(frameon=False, ncol=4, loc="upper left")
save(fig, "monthly_by_category.png")

## Stores: size and growth

Growth compares each store's last four weeks with its first four.

In [ ]:
per_week = sales.groupby(["store", "week"])["revenue"].sum().unstack()
growth = (per_week.iloc[:, -4:].sum(axis=1) / per_week.iloc[:, :4].sum(axis=1) - 1) * 100
by_store = (sales.groupby(["store", "city"])["revenue"].sum() / 1000).reset_index().set_index("store")
by_store["growth"] = growth
by_store = by_store.sort_values("revenue")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True, gridspec_kw={"width_ratios": [3, 2]})
labels = [f"{city} ({s})" for s, city in zip(by_store.index, by_store["city"])]
ax1.barh(labels, by_store["revenue"], color="C0")
ax1.set_xlabel("revenue (IDR millions)")
ax1.set_title("Revenue per store")
colors = ["C2" if g >= 0 else "C3" for g in by_store["growth"]]
ax2.barh(labels, by_store["growth"], color=colors)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_xlabel("growth (%)")
ax2.set_title("Last 4 weeks vs first 4")
save(fig, "stores.png")

## Each category's trend

In [ ]:
cats = sorted(sales["category"].unique())
fig, axes = plt.subplots(2, 2, figsize=(9, 5.5), sharex=True)
for ax, cat in zip(axes.flat, cats):
    s = sales[sales["category"] == cat].groupby("week")[["units", "revenue"]].sum()
    ax.fill_between(s.index, s["units"], alpha=0.25, color="C0")
    ax.plot(s.index, s["units"], color="C0")
    ax.set_title(f"{cat}: units per week")
    ax.tick_params(axis="x", labelrotation=30)
    avg = s["revenue"].sum() * 1000 / s["units"].sum()
    ax.text(0.98, 0.92, f"avg ticket IDR {avg:,.0f}", transform=ax.transAxes, ha="right", fontsize=8)
fig.tight_layout()
save(fig, "categories.png")

## The result

The caller gets the totals as JSON; `summary.csv` is kept as an artifact.

In [ ]:
by_province = (sales.groupby("province")["revenue"].sum() / 1000).round(1).sort_values(ascending=False)
summary = by_store.sort_values("revenue", ascending=False).round(1)
summary.to_csv(os.path.join(OUT, "summary.csv"))
print(summary.to_string())

nbsandbox.output({
    "weeks": int(sales["week"].nunique()),
    "revenue_millions": round(float(sales["revenue"].sum()) / 1000, 1),
    "by_province": by_province.to_dict(),
    "top_stores": summary.head(3).reset_index()[["store", "city", "revenue", "growth"]],
})